In [32]:
import pandas as pd
from fedot.core.data.data_detection import TextDataDetector
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import random
import uuid
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re
import string

import numpy as np
import pandas as pd

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

In [33]:
df = pd.DataFrame(
    {
        "name": [
            "Alex Smith",
            "John Brown",
            "Anna White",
            "Mike Green",
            "Kate Black",
        ],

        "url": [
            "https://example.com/a",
            "https://example.com/b",
            "https://example.com/c",
            "https://example.com/d",
            "https://example.com/e",
        ],

        "product_id": [
            "A123",
            "B234",
            "C345",
            "D456",
            "E567",
        ],
    }
)

detector = TextDataDetector()

print(detector.define_text_columns(df))

['name', 'url', 'product_id']


In [34]:
df = pd.DataFrame(
    {
        "review": [
            "I really liked this product",
            "The delivery was fast",
            "The quality was worse than expected",
            "Very comfortable for everyday use",
            "I have used it for several months",
        ],
        "name": [
            "Alex Smith",
            "John Brown",
            "Anna White",
            "Mike Green",
            "Kate Black",
        ],
        "url": [
            "https://example.com/a",
            "https://example.com/b",
            "https://example.com/c",
            "https://example.com/d",
            "https://example.com/e",
        ],
        "product_id": [
            "A123",
            "B234",
            "C345",
            "D456",
            "E567",
        ],
        "city": [
            "Moscow",
            "SPb",
            "Moscow",
            "Kazan",
            "SPb",
        ],
    }
)

detector = TextDataDetector()

print(detector.define_text_columns(df))

['review', 'name', 'url', 'product_id']


In [35]:
for column in df.columns:
    print(
        column,
        detector._column_contains_text(df[column]),
        df[column].nunique() / len(df[column]),
    )

review True 1.0
name True 1.0
url True 1.0
product_id True 1.0
city False 0.6


тут определил главную ошибку что текущая логика может ошибочно считать текстом строковые колонки с высокой долей уникальных значений, даже если они не содержат естественный текст

In [36]:
class Pr903TextDetector:
    UNIQUE_RATIO_THRESHOLD = 0.6
    MIN_VOCABULARY_SIZE = 20

    def contains_text(self, column: pd.Series) -> bool:
        if column.dtype != object:
            return False

        if self.is_float_compatible(column):
            return False

        if self.is_link(column):
            return False

        values = column.dropna().astype(str)

        if len(values) == 0:
            return False

        unique_ratio = values.nunique() / len(values)

        if unique_ratio <= self.UNIQUE_RATIO_THRESHOLD:
            return False

        vectorizer = TfidfVectorizer()

        try:
            vectorizer.fit(values)
        except ValueError:
            return False

        return len(vectorizer.vocabulary_) > self.MIN_VOCABULARY_SIZE

    @staticmethod
    def is_float_compatible(column):
        values = column.dropna()

        if len(values) == 0:
            return False

        converted = pd.to_numeric(values, errors="coerce")

        return converted.notna().mean() > 0.5

    @staticmethod
    def is_link(column):
        values = column.dropna().astype(str)

        if len(values) == 0:
            return False

        pattern = r"https?://"

        return values.str.contains(pattern, regex=True).mean() > 0.5

In [37]:
detector_pr = Pr903TextDetector()
detector_curr = TextDataDetector()

for column in df.columns:
    print(
        column,
        detector_pr.contains_text(df[column]),
        detector_curr._column_contains_text(df[column])
    )

review True True
name False True
url False True
product_id False True
city False False


In [38]:
import re
import string

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


def extract_column_features(column: pd.Series) -> dict:
    values = column.dropna().astype(str)

    if len(values) == 0:
        return {}

    # базовые размеры строк
    lengths = values.str.len()

    # количество слов
    word_counts = values.apply(
        lambda x: len(re.findall(r"\b\w+\b", x))
    )

    # все токены во всей колонке
    tokens = []

    for value in values:
        tokens.extend(
            re.findall(r"\b\w+\b", value.lower())
        )

    unique_tokens = set(tokens)

    # все символы колонки
    full_text = "".join(values)

    total_chars = len(full_text)

    if total_chars == 0:
        total_chars = 1

    alphabetic_chars = sum(char.isalpha() for char in full_text)
    digit_chars = sum(char.isdigit() for char in full_text)
    whitespace_chars = sum(char.isspace() for char in full_text)
    punctuation_chars = sum(char in string.punctuation for char in full_text)

    # специальные типы значений
    url_pattern = re.compile(r"^https?://", re.IGNORECASE)

    email_pattern = re.compile(
        r"^[\w\.-]+@[\w\.-]+\.\w+$"
    )

    uuid_pattern = re.compile(
        r"^[0-9a-fA-F]{8}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{12}$"
    )

    numeric_pattern = re.compile(
        r"^[+-]?(\d+(\.\d*)?|\.\d+)$"
    )

    url_ratio = values.apply(
        lambda x: bool(url_pattern.search(x))
    ).mean()

    email_ratio = values.apply(
        lambda x: bool(email_pattern.match(x))
    ).mean()

    uuid_ratio = values.apply(
        lambda x: bool(uuid_pattern.match(x))
    ).mean()

    numeric_ratio = values.apply(
        lambda x: bool(numeric_pattern.match(x))
    ).mean()

    # tf-idf vocabulary
    try:
        vectorizer = TfidfVectorizer()
        vectorizer.fit(values)

        tfidf_vocabulary_size = len(
            vectorizer.vocabulary_
        )

    except ValueError:
        tfidf_vocabulary_size = 0

    total_tokens = len(tokens)

    vocabulary_ratio = (
        len(unique_tokens) / total_tokens
        if total_tokens > 0
        else 0
    )

    mean_token_length = (
        np.mean([len(token) for token in tokens])
        if tokens
        else 0
    )

    return {
        "rows": len(values),

        "unique_ratio":
            values.nunique() / len(values),

        "mean_length":
            lengths.mean(),

        "median_length":
            lengths.median(),

        "std_length":
            lengths.std(),

        "mean_word_count":
            word_counts.mean(),

        "median_word_count":
            word_counts.median(),

        "std_word_count":
            word_counts.std(),

        "single_word_ratio":
            (word_counts == 1).mean(),

        "multi_word_ratio":
            (word_counts > 1).mean(),

        "vocabulary_size":
            len(unique_tokens),

        "vocabulary_ratio":
            vocabulary_ratio,

        "tfidf_vocabulary_size":
            tfidf_vocabulary_size,

        "mean_token_length":
            mean_token_length,

        "alphabetic_ratio":
            alphabetic_chars / total_chars,

        "digit_ratio":
            digit_chars / total_chars,

        "whitespace_ratio":
            whitespace_chars / total_chars,

        "punctuation_ratio":
            punctuation_chars / total_chars,

        "url_ratio":
            url_ratio,

        "email_ratio":
            email_ratio,

        "uuid_ratio":
            uuid_ratio,

        "numeric_ratio":
            numeric_ratio,
    }

In [39]:
features = {}

for column in df.columns:
    features[column] = extract_column_features(
        df[column]
    )

features_df = pd.DataFrame(features).T

features_df

,rows,unique_ratio,mean_length,median_length,std_length,mean_word_count,median_word_count,std_word_count,single_word_ratio,multi_word_ratio,...,tfidf_vocabulary_size,mean_token_length,alphabetic_ratio,digit_ratio,whitespace_ratio,punctuation_ratio,url_ratio,email_ratio,uuid_ratio,numeric_ratio
review,5.0,1.0,29.8,33.0,5.761944,5.4,5.0,1.140175,0.0,1.0,...,22.0,4.703704,0.852349,0.00,0.147651,0.000000,0.0,0.0,0.0,0.0
name,5.0,1.0,10.0,10.0,0.000000,2.0,2.0,0.000000,0.0,1.0,...,10.0,4.500000,0.900000,0.00,0.100000,0.000000,0.0,0.0,0.0,0.0
url,5.0,1.0,21.0,21.0,0.000000,4.0,4.0,0.000000,0.0,1.0,...,3.0,4.000000,0.761905,0.00,0.000000,0.238095,1.0,0.0,0.0,0.0
product_id,5.0,1.0,4.0,4.0,0.000000,1.0,1.0,0.000000,1.0,0.0,...,5.0,4.000000,0.250000,0.75,0.000000,0.000000,0.0,0.0,0.0,0.0
city,5.0,0.6,4.6,5.0,1.516575,1.0,1.0,0.000000,1.0,0.0,...,3.0,4.600000,1.000000,0.00,0.000000,0.000000,0.0,0.0,0.0,0.0


In [40]:
features_df["current_fedot"] = [
    detector_curr._column_contains_text(df[column])
    for column in df.columns
]

features_df["pr903"] = [
    detector_pr.contains_text(df[column])
    for column in df.columns
]

In [41]:
features_df[
    [
        "unique_ratio",
        "mean_length",
        "mean_word_count",
        "single_word_ratio",
        "vocabulary_size",
        "vocabulary_ratio",
        "alphabetic_ratio",
        "digit_ratio",
        "whitespace_ratio",
        "punctuation_ratio",
        "url_ratio",
        "email_ratio",
        "uuid_ratio",
    ]
]

,unique_ratio,mean_length,mean_word_count,single_word_ratio,vocabulary_size,vocabulary_ratio,alphabetic_ratio,digit_ratio,whitespace_ratio,punctuation_ratio,url_ratio,email_ratio,uuid_ratio
review,1.0,29.8,5.4,0.0,23.0,0.851852,0.852349,0.00,0.147651,0.000000,0.0,0.0,0.0
name,1.0,10.0,2.0,0.0,10.0,1.000000,0.900000,0.00,0.100000,0.000000,0.0,0.0,0.0
url,1.0,21.0,4.0,0.0,8.0,0.400000,0.761905,0.00,0.000000,0.238095,1.0,0.0,0.0
product_id,1.0,4.0,1.0,1.0,5.0,1.000000,0.250000,0.75,0.000000,0.000000,0.0,0.0,0.0
city,0.6,4.6,1.0,1.0,3.0,0.600000,1.000000,0.00,0.000000,0.000000,0.0,0.0,0.0


In [42]:
import random
import uuid

n = 200

df_hard = pd.DataFrame({
    # настоящий текст
    "reviews": [
        f"this product is very useful and works well for everyday tasks number {i}"
        for i in range(n)
    ],

    # короткий текст
    "titles": [
        f"machine learning experiment number {i}"
        for i in range(n)
    ],

    # имена: большой словарь, но это не text
    "names": [
        f"user firstname{i} lastname{i}"
        for i in range(n)
    ],

    # названия товаров
    "product_names": [
        f"Samsung Galaxy Model {i}"
        for i in range(n)
    ],

    # id
    "product_ids": [
        f"PRD-{i:06d}"
        for i in range(n)
    ],

    # uuid
    "uuid": [
        str(uuid.uuid4())
        for _ in range(n)
    ],

    # email
    "email": [
        f"user{i}@example.com"
        for i in range(n)
    ],

    # url
    "url": [
        f"https://example.com/product/{i}"
        for i in range(n)
    ],

    # категориальный признак
    "city": [
        random.choice(["Moscow", "SPb", "Kazan", "Omsk"])
        for _ in range(n)
    ],

    # файловые пути
    "filepath": [
        f"/home/user/files/document_{i}.txt"
        for i in range(n)
    ],
})

In [43]:
for column in df_hard.columns:
    print(
        f"{column:15}",
        "current:",
        detector_curr._column_contains_text(df_hard[column]),
        "pr903:",
        detector_pr.contains_text(df_hard[column]),
    )

reviews         current: True pr903: True
titles          current: True pr903: True
names           current: True pr903: True
product_names   current: True pr903: True
product_ids     current: True pr903: True
uuid            current: True pr903: True
email           current: True pr903: True
url             current: True pr903: False
city            current: False pr903: False
filepath        current: True pr903: True


In [44]:
features = {
    column: extract_column_features(df_hard[column])
    for column in df_hard.columns
}

features_df = pd.DataFrame(features).T

In [45]:
features_df[
    [
        "unique_ratio",
        "mean_length",
        "mean_word_count",
        "single_word_ratio",
        "vocabulary_size",
        "vocabulary_ratio",
        "digit_ratio",
        "whitespace_ratio",
        "punctuation_ratio",
        "url_ratio",
        "email_ratio",
        "uuid_ratio",
    ]
]
ground_truth = {
    "reviews": True,
    "titles": True,

    "names": False,
    "product_names": False,
    "product_ids": False,
    "uuid": False,
    "email": False,
    "url": False,
    "city": False,
    "filepath": False,
}
results = []

for column in df_hard.columns:
    results.append({
        "column": column,
        "expected": ground_truth[column],

        "current": detector_curr._column_contains_text(
            df_hard[column]
        ),

        "pr903": detector_pr.contains_text(
            df_hard[column]
        ),
    })

results_df = pd.DataFrame(results)

results_df

,column,expected,current,pr903
0,reviews,True,True,True
1,titles,True,True,True
2,names,False,True,True
3,product_names,False,True,True
4,product_ids,False,True,True
5,uuid,False,True,True
6,email,False,True,True
7,url,False,True,False
8,city,False,False,False
9,filepath,False,True,True


In [46]:
for detector_name in ["current", "pr903"]:
    y_true = results_df["expected"]
    y_pred = results_df[detector_name]

    print(detector_name)
    print("accuracy:", accuracy_score(y_true, y_pred))
    print("precision:", precision_score(y_true, y_pred))
    print("recall:", recall_score(y_true, y_pred))
    print("f1:", f1_score(y_true, y_pred))
    print()

current
accuracy: 0.3
precision: 0.2222222222222222
recall: 1.0
f1: 0.36363636363636365

pr903
accuracy: 0.4
precision: 0.25
recall: 1.0
f1: 0.4



In [47]:
def get_pattern_ratios(column: pd.Series):
    values = column.dropna().astype(str)

    if len(values) == 0:
        return {}

    url_pattern = re.compile(r"^https?://", re.I)

    email_pattern = re.compile(
        r"^[\w.\-+]+@[\w.\-]+\.\w+$"
    )

    uuid_pattern = re.compile(
        r"^[0-9a-fA-F]{8}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{12}$"
    )

    path_pattern = re.compile(
        r"^(?:[A-Za-z]:[\\/]|/).+"
    )

    id_pattern = re.compile(
        r"^[A-Za-z]{0,10}[-_]?\d+[A-Za-z0-9_-]*$"
    )

    return {
        "url_ratio": values.apply(
            lambda x: bool(url_pattern.match(x))
        ).mean(),

        "email_ratio": values.apply(
            lambda x: bool(email_pattern.match(x))
        ).mean(),

        "uuid_ratio": values.apply(
            lambda x: bool(uuid_pattern.match(x))
        ).mean(),

        "path_ratio": values.apply(
            lambda x: bool(path_pattern.match(x))
        ).mean(),

        "id_ratio": values.apply(
            lambda x: bool(id_pattern.match(x))
        ).mean(),
    }

In [48]:
def normalize_value(value: str) -> str:
    value = re.sub(r"\d+", "#", value)
    value = re.sub(r"[a-fA-F0-9]{8,}", "<hex>", value)

    return value


def normalized_pattern_ratio(column: pd.Series) -> float:
    values = column.dropna().astype(str)

    if len(values) == 0:
        return 0

    normalized = values.apply(normalize_value)

    return normalized.nunique() / len(normalized)

In [49]:
class ImprovedTextDetector:

    def contains_text(self, column: pd.Series) -> bool:
        values = column.dropna().astype(str)

        if len(values) == 0:
            return False

        patterns = get_pattern_ratios(column)

        # очевидные специальные типы
        if patterns["url_ratio"] > 0.8:
            return False

        if patterns["email_ratio"] > 0.8:
            return False

        if patterns["uuid_ratio"] > 0.8:
            return False

        if patterns["path_ratio"] > 0.8:
            return False

        if patterns["id_ratio"] > 0.8:
            return False

        word_counts = values.apply(
            lambda x: len(re.findall(r"\b\w+\b", x))
        )

        lengths = values.str.len()

        mean_words = word_counts.mean()
        median_words = word_counts.median()
        mean_length = lengths.mean()

        unique_ratio = values.nunique() / len(values)

        # категории
        if unique_ratio < 0.6:
            return False

        # длинный естественный текст
        if (
            mean_words >= 4
            and mean_length >= 20
        ):
            return True

        return False

In [50]:
detector_improved = ImprovedTextDetector()

results = []

for column in df_hard.columns:
    results.append({
        "column": column,
        "expected": ground_truth[column],

        "current":
            detector_curr._column_contains_text(
                df_hard[column]
            ),

        "pr903":
            detector_pr.contains_text(
                df_hard[column]
            ),

        "improved":
            detector_improved.contains_text(
                df_hard[column]
            ),
    })

results_df = pd.DataFrame(results)

results_df

,column,expected,current,pr903,improved
0,reviews,True,True,True,True
1,titles,True,True,True,True
2,names,False,True,True,False
3,product_names,False,True,True,True
4,product_ids,False,True,True,False
5,uuid,False,True,True,False
6,email,False,True,True,False
7,url,False,True,False,False
8,city,False,False,False,False
9,filepath,False,True,True,False


In [51]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


for detector_name in [
    "current",
    "pr903",
    "improved",
]:
    y_true = results_df["expected"]
    y_pred = results_df[detector_name]

    print(detector_name)

    print(
        "accuracy:",
        accuracy_score(y_true, y_pred)
    )

    print(
        "precision:",
        precision_score(y_true, y_pred)
    )

    print(
        "recall:",
        recall_score(y_true, y_pred)
    )

    print(
        "f1:",
        f1_score(y_true, y_pred)
    )

    print()

current
accuracy: 0.3
precision: 0.2222222222222222
recall: 1.0
f1: 0.36363636363636365

pr903
accuracy: 0.4
precision: 0.25
recall: 1.0
f1: 0.4

improved
accuracy: 0.9
precision: 0.6666666666666666
recall: 1.0
f1: 0.8



In [52]:
results_df[
    results_df["expected"] != results_df["improved"]
]
wrong_columns = results_df[
    results_df["expected"] != results_df["improved"]
]["column"]

features_df.loc[wrong_columns]

,rows,unique_ratio,mean_length,median_length,std_length,mean_word_count,median_word_count,std_word_count,single_word_ratio,multi_word_ratio,...,tfidf_vocabulary_size,mean_token_length,alphabetic_ratio,digit_ratio,whitespace_ratio,punctuation_ratio,url_ratio,email_ratio,uuid_ratio,numeric_ratio
product_names,200.0,1.0,23.45,23.5,0.590971,4.0,4.0,0.0,0.0,1.0,...,193.0,5.1125,0.767591,0.104478,0.127932,0.0,0.0,0.0,0.0,0.0


In [53]:
features_df.loc[
    ["reviews", "titles", *wrong_columns]
][[
    "mean_length",
    "mean_word_count",
    "unique_ratio",
    "vocabulary_ratio",
    "digit_ratio",
    "whitespace_ratio",
    "punctuation_ratio",
]]

,mean_length,mean_word_count,unique_ratio,vocabulary_ratio,digit_ratio,whitespace_ratio,punctuation_ratio
reviews,71.45,13.0,1.0,0.081538,0.034290,0.167950,0.0
titles,37.45,5.0,1.0,0.204000,0.065421,0.106809,0.0
product_names,23.45,4.0,1.0,0.253750,0.104478,0.127932,0.0


In [54]:
def common_token_ratio(column: pd.Series) -> float:
    values = column.dropna().astype(str)

    if len(values) == 0:
        return 0.0

    tokens_per_row = [
        re.findall(r"\b\w+\b", value.lower())
        for value in values
    ]

    all_tokens = [
        token
        for row in tokens_per_row
        for token in row
    ]

    if not all_tokens:
        return 0.0

    counts = Counter(all_tokens)

    repeated_tokens = sum(
        count
        for count in counts.values()
        if count > 1
    )

    return repeated_tokens / len(all_tokens)


def normalize_pattern(value: str) -> str:
    value = value.lower()

    # uuid
    value = re.sub(
        r"[0-9a-f]{8}-"
        r"[0-9a-f]{4}-"
        r"[0-9a-f]{4}-"
        r"[0-9a-f]{4}-"
        r"[0-9a-f]{12}",
        "<uuid>",
        value,
    )

    # числа
    value = re.sub(
        r"\d+",
        "<num>",
        value,
    )

    return value


def pattern_unique_ratio(column: pd.Series) -> float:
    values = column.dropna().astype(str)

    if len(values) == 0:
        return 0.0

    patterns = values.apply(normalize_pattern)

    return patterns.nunique() / len(patterns)


def extract_column_features(column: pd.Series) -> dict:
    values = column.dropna().astype(str)

    if len(values) == 0:
        return {}

    lengths = values.str.len()

    word_counts = values.apply(
        lambda x: len(
            re.findall(r"\b\w+\b", x)
        )
    )

    tokens = []

    for value in values:
        tokens.extend(
            re.findall(
                r"\b\w+\b",
                value.lower(),
            )
        )

    unique_tokens = set(tokens)

    full_text = "".join(values)

    total_chars = len(full_text)

    if total_chars == 0:
        total_chars = 1

    alphabetic_chars = sum(
        char.isalpha()
        for char in full_text
    )

    digit_chars = sum(
        char.isdigit()
        for char in full_text
    )

    whitespace_chars = sum(
        char.isspace()
        for char in full_text
    )

    punctuation_chars = sum(
        char in string.punctuation
        for char in full_text
    )

    url_pattern = re.compile(
        r"^https?://",
        re.IGNORECASE,
    )

    email_pattern = re.compile(
        r"^[\w.\-+]+@[\w.\-]+\.\w+$"
    )

    uuid_pattern = re.compile(
        r"^[0-9a-fA-F]{8}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{4}-"
        r"[0-9a-fA-F]{12}$"
    )

    numeric_pattern = re.compile(
        r"^[+-]?(\d+(\.\d*)?|\.\d+)$"
    )

    path_pattern = re.compile(
        r"^(?:[A-Za-z]:[\\/]|/).+"
    )

    id_pattern = re.compile(
        r"^[A-Za-z]{0,10}[-_]?\d+[A-Za-z0-9_-]*$"
    )

    url_ratio = values.apply(
        lambda x: bool(
            url_pattern.match(x)
        )
    ).mean()

    email_ratio = values.apply(
        lambda x: bool(
            email_pattern.match(x)
        )
    ).mean()

    uuid_ratio = values.apply(
        lambda x: bool(
            uuid_pattern.match(x)
        )
    ).mean()

    numeric_ratio = values.apply(
        lambda x: bool(
            numeric_pattern.match(x)
        )
    ).mean()

    path_ratio = values.apply(
        lambda x: bool(
            path_pattern.match(x)
        )
    ).mean()

    id_ratio = values.apply(
        lambda x: bool(
            id_pattern.match(x)
        )
    ).mean()

    try:
        vectorizer = TfidfVectorizer()

        vectorizer.fit(values)

        tfidf_vocabulary_size = len(
            vectorizer.vocabulary_
        )

    except ValueError:
        tfidf_vocabulary_size = 0

    total_tokens = len(tokens)

    vocabulary_ratio = (
        len(unique_tokens) / total_tokens
        if total_tokens > 0
        else 0.0
    )

    mean_token_length = (
        np.mean(
            [len(token) for token in tokens]
        )
        if tokens
        else 0.0
    )

    return {
        "rows":
            len(values),

        "unique_ratio":
            values.nunique() / len(values),

        "mean_length":
            lengths.mean(),

        "median_length":
            lengths.median(),

        "std_length":
            lengths.std(),

        "mean_word_count":
            word_counts.mean(),

        "median_word_count":
            word_counts.median(),

        "std_word_count":
            word_counts.std(),

        "single_word_ratio":
            (word_counts == 1).mean(),

        "multi_word_ratio":
            (word_counts > 1).mean(),

        "vocabulary_size":
            len(unique_tokens),

        "vocabulary_ratio":
            vocabulary_ratio,

        "tfidf_vocabulary_size":
            tfidf_vocabulary_size,

        "mean_token_length":
            mean_token_length,

        "alphabetic_ratio":
            alphabetic_chars / total_chars,

        "digit_ratio":
            digit_chars / total_chars,

        "whitespace_ratio":
            whitespace_chars / total_chars,

        "punctuation_ratio":
            punctuation_chars / total_chars,

        "url_ratio":
            url_ratio,

        "email_ratio":
            email_ratio,

        "uuid_ratio":
            uuid_ratio,

        "numeric_ratio":
            numeric_ratio,

        "path_ratio":
            path_ratio,

        "id_ratio":
            id_ratio,

        "common_token_ratio":
            common_token_ratio(column),

        "pattern_unique_ratio":
            pattern_unique_ratio(column),
    }

In [55]:
features = {
    column: extract_column_features(
        df_hard[column]
    )
    for column in df_hard.columns
}

features_df = pd.DataFrame(features).T

features_df

,rows,unique_ratio,mean_length,median_length,std_length,mean_word_count,median_word_count,std_word_count,single_word_ratio,multi_word_ratio,...,whitespace_ratio,punctuation_ratio,url_ratio,email_ratio,uuid_ratio,numeric_ratio,path_ratio,id_ratio,common_token_ratio,pattern_unique_ratio
reviews,200.0,1.00,71.45,71.5,0.590971,13.0,13.0,0.0,0.0,1.0,...,0.167950,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.923077,0.005
titles,200.0,1.00,37.45,37.5,0.590971,5.0,5.0,0.0,0.0,1.0,...,0.106809,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.800000,0.005
names,200.0,1.00,27.90,28.0,1.181941,3.0,3.0,0.0,0.0,1.0,...,0.071685,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.005
product_names,200.0,1.00,23.45,23.5,0.590971,4.0,4.0,0.0,0.0,1.0,...,0.127932,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.750000,0.005
product_ids,200.0,1.00,10.00,10.0,0.000000,2.0,2.0,0.0,0.0,1.0,...,0.000000,0.100000,0.0,0.0,0.0,0.0,0.0,1.0,0.500000,0.005
uuid,200.0,1.00,36.00,36.0,0.000000,5.0,5.0,0.0,0.0,1.0,...,0.000000,0.111111,0.0,0.0,1.0,0.0,0.0,1.0,0.025000,0.005
email,200.0,1.00,18.45,18.5,0.590971,3.0,3.0,0.0,0.0,1.0,...,0.000000,0.108401,0.0,1.0,0.0,0.0,0.0,0.0,0.666667,0.005
url,200.0,1.00,30.45,30.5,0.590971,5.0,5.0,0.0,0.0,1.0,...,0.000000,0.197044,1.0,0.0,0.0,0.0,0.0,0.0,0.800000,0.005
city,200.0,0.02,4.49,5.0,1.098148,1.0,1.0,0.0,1.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.020
filepath,200.0,1.00,32.45,32.5,0.590971,5.0,5.0,0.0,0.0,1.0,...,0.000000,0.184900,0.0,0.0,0.0,0.0,1.0,0.0,0.800000,0.005


In [56]:
features_df.loc[
    [
        "reviews",
        "titles",
        "product_names",
    ],
    [
        "mean_length",
        "mean_word_count",
        "vocabulary_ratio",
        "common_token_ratio",
        "pattern_unique_ratio",
        "digit_ratio",
        "whitespace_ratio",
        "punctuation_ratio",
    ]
]

,mean_length,mean_word_count,vocabulary_ratio,common_token_ratio,pattern_unique_ratio,digit_ratio,whitespace_ratio,punctuation_ratio
reviews,71.45,13.0,0.081538,0.923077,0.005,0.034290,0.167950,0.0
titles,37.45,5.0,0.204000,0.800000,0.005,0.065421,0.106809,0.0
product_names,23.45,4.0,0.253750,0.750000,0.005,0.104478,0.127932,0.0


In [57]:
df_users = pd.DataFrame({
    "bio": [
        "python developer interested in machine learning",
        "student studying artificial intelligence and data science",
        "software engineer working with backend systems",
        "researcher interested in neural networks",
        "data analyst working with large datasets",
    ],

    "name": [
        "Alex Smith",
        "John Brown",
        "Anna White",
        "Mike Green",
        "Kate Black",
    ],

    "email": [
        "alex@example.com",
        "john@example.com",
        "anna@example.com",
        "mike@example.com",
        "kate@example.com",
    ],

    "city": [
        "London",
        "Moscow",
        "London",
        "Berlin",
        "Moscow",
    ],
})


labels_users = {
    "bio": 1,
    "name": 0,
    "email": 0,
    "city": 0,
}

df_products = pd.DataFrame({
    "review": [
        "the product is very good and works perfectly",
        "delivery was fast and the quality is excellent",
        "I have been using this product every day",
        "the device is comfortable and easy to use",
        "I would definitely recommend this product",
    ],

    "product_name": [
        "Samsung Galaxy S25",
        "Apple iPhone 17",
        "Sony WH1000XM6",
        "MacBook Pro",
        "Asus Zenbook",
    ],

    "product_id": [
        "PRD-001",
        "PRD-002",
        "PRD-003",
        "PRD-004",
        "PRD-005",
    ],

    "category": [
        "phone",
        "phone",
        "headphones",
        "laptop",
        "laptop",
    ],
})


labels_products = {
    "review": 1,
    "product_name": 0,
    "product_id": 0,
    "category": 0,
}

df_articles = pd.DataFrame({
    "title": [
        "Machine learning methods for text classification",
        "New approaches to neural network optimisation",
        "How artificial intelligence changes software development",
        "Modern methods of time series forecasting",
        "Automatic machine learning systems",
    ],

    "description": [
        "This article describes several methods for training machine learning models.",
        "The paper studies optimisation techniques for deep neural networks.",
        "The article discusses applications of artificial intelligence in software engineering.",
        "Different forecasting algorithms are compared using real datasets.",
        "The work describes methods for automatic construction of machine learning pipelines.",
    ],

    "url": [
        "https://example.com/1",
        "https://example.com/2",
        "https://example.com/3",
        "https://example.com/4",
        "https://example.com/5",
    ],

    "author": [
        "John Smith",
        "Anna Brown",
        "Mike Wilson",
        "Kate Green",
        "Alex White",
    ],
})


labels_articles = {
    "title": 1,
    "description": 1,
    "url": 0,
    "author": 0,
}
datasets = [
    (
        "products",
        df_products,
        labels_products,
    ),

    (
        "users",
        df_users,
        labels_users,
    ),

    (
        "articles",
        df_articles,
        labels_articles,
    ),
]

In [58]:
rows = []

for dataset_name, dataframe, labels in datasets:
    for column_name, target in labels.items():

        features = extract_column_features(
            dataframe[column_name]
        )

        features["dataset"] = dataset_name
        features["column"] = column_name
        features["target"] = target

        rows.append(features)

feature_dataset = pd.DataFrame(rows)

feature_dataset

,rows,unique_ratio,mean_length,median_length,std_length,mean_word_count,median_word_count,std_word_count,single_word_ratio,multi_word_ratio,...,email_ratio,uuid_ratio,numeric_ratio,path_ratio,id_ratio,common_token_ratio,pattern_unique_ratio,dataset,column,target
0,5,1.0,42.4,41.0,2.509980,7.6,8.0,0.894427,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.421053,1.0,products,review,1
1,5,1.0,14.0,14.0,2.738613,2.4,2.0,0.547723,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,1.0,products,product_name,0
2,5,1.0,7.0,7.0,0.000000,2.0,2.0,0.000000,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.500000,0.2,products,product_id,0
3,5,0.6,6.4,6.0,2.073644,1.0,1.0,0.000000,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.800000,0.6,products,category,0
4,5,1.0,46.0,46.0,6.964194,6.0,6.0,0.707107,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.333333,1.0,users,bio,1
5,5,1.0,10.0,10.0,0.000000,2.0,2.0,0.000000,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,1.0,users,name,0
6,5,1.0,16.0,16.0,0.000000,3.0,3.0,0.000000,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.666667,1.0,users,email,0
7,5,0.6,6.0,6.0,0.000000,1.0,1.0,0.000000,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.800000,0.6,users,city,0
8,5,1.0,44.8,45.0,8.167007,5.6,6.0,0.894427,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.214286,1.0,articles,title,1
9,5,1.0,75.8,76.0,9.284396,9.6,10.0,1.140175,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.375000,1.0,articles,description,1


In [59]:
feature_columns = [
    "unique_ratio",
    "mean_length",
    "median_length",
    "std_length",
    "mean_word_count",
    "median_word_count",
    "std_word_count",
    "single_word_ratio",
    "multi_word_ratio",
    "vocabulary_size",
    "vocabulary_ratio",
    "tfidf_vocabulary_size",
    "mean_token_length",
    "alphabetic_ratio",
    "digit_ratio",
    "whitespace_ratio",
    "punctuation_ratio",
    "url_ratio",
    "email_ratio",
    "uuid_ratio",
    "numeric_ratio",
    "path_ratio",
    "id_ratio",
    "common_token_ratio",
    "pattern_unique_ratio",
]

X = feature_dataset[feature_columns].fillna(0)
y = feature_dataset["target"]

In [ ]:
class ImprovedTextDataDetector(TextDataDetector):
    STRUCTURED_THRESHOLD = 0.8
    MIN_UNIQUE_RATIO = 0.3
    MIN_VALUES = 3

    def _column_contains_text(
        self,
        column: pd.Series,
    ) -> bool:

        # только строковые колонки
        if not (
            pd.api.types.is_object_dtype(column)
            or pd.api.types.is_string_dtype(column)
        ):
            return False

        values = (
            column
            .dropna()
            .astype(str)
            .str.strip()
        )

        values = values[
            values != ""
        ]

        if len(values) < self.MIN_VALUES:
            return False

        # строки, которые на самом деле являются числами
        if self._is_float_compatible(column):
            return False

        features = self.extract_features(
            values
        )

        # явные структурированные данные
        structured_features = [
            "url_ratio",
            "email_ratio",
            "uuid_ratio",
            "path_ratio",
            "numeric_ratio",
            "id_ratio",
        ]

        for feature in structured_features:
            if (
                features[feature]
                >= self.STRUCTURED_THRESHOLD
            ):
                return False

        # обычная категориальная колонка
        if (
            features["unique_ratio"]
            < self.MIN_UNIQUE_RATIO
        ):
            return False

        # длинный естественный текст
        if (
            features["median_word_count"] >= 7
            and features["median_length"] >= 35
            and features["alphabetic_ratio"] >= 0.55
        ):
            return True

        # обычные предложения / описания / отзывы
        if (
            features["mean_word_count"] >= 5
            and features["mean_length"] >= 25
            and features["alphabetic_ratio"] >= 0.60
            and features["digit_ratio"] < 0.25
            and features["pattern_unique_ratio"] >= 0.15
        ):
            return True

        # короткий естественный текст:
        # заголовки, bio и т.п.
        if (
            features["mean_word_count"] >= 4
            and features["mean_length"] >= 20
            and features["alphabetic_ratio"] >= 0.70
            and features["digit_ratio"] < 0.15
            and features["function_word_ratio"] >= 0.04
        ):
            return True

        return False

    def extract_features(
        self,
        values: pd.Series,
    ) -> dict:

        lengths = values.str.len()

        word_lists = values.apply(
            lambda value: re.findall(
                r"\b\w+\b",
                value.lower(),
            )
        )

        word_counts = word_lists.apply(
            len
        )

        tokens = [
            token
            for words in word_lists
            for token in words
        ]

        unique_tokens = set(tokens)

        full_text = "".join(
            values.tolist()
        )

        total_chars = max(
            len(full_text),
            1,
        )

        alphabetic_chars = sum(
            char.isalpha()
            for char in full_text
        )

        digit_chars = sum(
            char.isdigit()
            for char in full_text
        )

        whitespace_chars = sum(
            char.isspace()
            for char in full_text
        )

        punctuation_chars = sum(
            char in string.punctuation
            for char in full_text
        )

        features = {
            "rows":
                len(values),

            "unique_ratio":
                values.nunique()
                / len(values),

            "mean_length":
                lengths.mean(),

            "median_length":
                lengths.median(),

            "std_length":
                self._safe_std(lengths),

            "mean_word_count":
                word_counts.mean(),

            "median_word_count":
                word_counts.median(),

            "std_word_count":
                self._safe_std(word_counts),

            "single_word_ratio":
                (word_counts == 1).mean(),

            "multi_word_ratio":
                (word_counts > 1).mean(),

            "alphabetic_ratio":
                alphabetic_chars
                / total_chars,

            "digit_ratio":
                digit_chars
                / total_chars,

            "whitespace_ratio":
                whitespace_chars
                / total_chars,

            "punctuation_ratio":
                punctuation_chars
                / total_chars,

            "vocabulary_size":
                len(unique_tokens),

            "vocabulary_ratio":
                (
                    len(unique_tokens)
                    / len(tokens)
                    if tokens
                    else 0.0
                ),

            "mean_token_length":
                (
                    np.mean([
                        len(token)
                        for token in tokens
                    ])
                    if tokens
                    else 0.0
                ),

            "common_token_ratio":
                self._common_token_ratio(
                    tokens
                ),

            "pattern_unique_ratio":
                self._pattern_unique_ratio(
                    values
                ),

            "function_word_ratio":
                self._function_word_ratio(
                    tokens
                ),
        }

        features.update(
            self._structured_ratios(
                values
            )
        )

        features[
            "tfidf_vocabulary_size"
        ] = self._tfidf_vocabulary_size(
            values
        )

        return features

    @staticmethod
    def _safe_std(
        values: pd.Series,
    ) -> float:

        result = values.std()

        if pd.isna(result):
            return 0.0

        return float(result)

    @staticmethod
    def _structured_ratios(
        values: pd.Series,
    ) -> dict:

        patterns = {
            "url_ratio": re.compile(
                r"^https?://",
                re.IGNORECASE,
            ),

            "email_ratio": re.compile(
                r"^[\w.\-+]+@"
                r"[\w.\-]+\.\w+$"
            ),

            "uuid_ratio": re.compile(
                r"^[0-9a-fA-F]{8}-"
                r"[0-9a-fA-F]{4}-"
                r"[0-9a-fA-F]{4}-"
                r"[0-9a-fA-F]{4}-"
                r"[0-9a-fA-F]{12}$"
            ),

            "numeric_ratio": re.compile(
                r"^[+-]?"
                r"(?:\d+(?:\.\d*)?"
                r"|\.\d+)$"
            ),

            "path_ratio": re.compile(
                r"^(?:"
                r"[A-Za-z]:[\\/]"
                r"|/"
                r").+"
            ),

            "id_ratio": re.compile(
                r"^[A-Za-z]{0,10}"
                r"[-_]?"
                r"\d+"
                r"[A-Za-z0-9_-]*$"
            ),
        }

        result = {}

        for name, pattern in patterns.items():
            result[name] = values.apply(
                lambda value: bool(
                    pattern.match(value)
                )
            ).mean()

        return result

    @staticmethod
    def _normalize_pattern(
        value: str,
    ) -> str:

        value = value.lower()

        # uuid
        value = re.sub(
            r"[0-9a-f]{8}-"
            r"[0-9a-f]{4}-"
            r"[0-9a-f]{4}-"
            r"[0-9a-f]{4}-"
            r"[0-9a-f]{12}",
            "<uuid>",
            value,
        )

        # числа
        value = re.sub(
            r"\d+",
            "<num>",
            value,
        )

        return value

    @classmethod
    def _pattern_unique_ratio(
        cls,
        values: pd.Series,
    ) -> float:

        patterns = values.apply(
            cls._normalize_pattern
        )

        return (
            patterns.nunique()
            / len(patterns)
        )

    @staticmethod
    def _common_token_ratio(
        tokens: list,
    ) -> float:

        if not tokens:
            return 0.0

        counts = Counter(tokens)

        repeated = sum(
            count
            for count in counts.values()
            if count > 1
        )

        return (
            repeated
            / len(tokens)
        )

    @staticmethod
    def _function_word_ratio(
        tokens: list,
    ) -> float:

        if not tokens:
            return 0.0

        # небольшой набор служебных слов
        # нужен только как дополнительный сигнал
        function_words = {
            "the", "a", "an",
            "and", "or",
            "of", "to", "in",
            "for", "with",
            "on", "at",
            "from", "by",
            "is", "are",
            "was", "were",
            "this", "that",

            "и", "или",
            "в", "во",
            "на", "с", "со",
            "для", "из",
            "от", "до",
            "по", "к", "у",
            "о", "об",
            "что", "как",
            "это", "не",
        }

        count = sum(
            token in function_words
            for token in tokens
        )

        return count / len(tokens)

    @staticmethod
    def _tfidf_vocabulary_size(
        values: pd.Series,
    ) -> int:

        try:
            vectorizer = (
                TfidfVectorizer()
            )

            vectorizer.fit(values)

            return len(
                vectorizer.vocabulary_
            )

        except ValueError:
            return 0

In [61]:
detector_final = (
    ImprovedTextDataDetector()
)

for column in df_hard.columns:
    print(
        f"{column:20}",
        detector_final
        ._column_contains_text(
            df_hard[column]
        ),
    )

reviews              True
titles               False
names                False
product_names        False
product_ids          False
uuid                 False
email                False
url                  False
city                 False
filepath             False


In [62]:
results = []

for column in df_hard.columns:
    results.append({
        "column":
            column,

        "expected":
            ground_truth[column],

        "current":
            detector_curr
            ._column_contains_text(
                df_hard[column]
            ),

        "pr903":
            detector_pr
            .contains_text(
                df_hard[column]
            ),

        "final":
            detector_final
            ._column_contains_text(
                df_hard[column]
            ),
    })

results_df = pd.DataFrame(
    results
)

results_df

,column,expected,current,pr903,final
0,reviews,True,True,True,True
1,titles,True,True,True,False
2,names,False,True,True,False
3,product_names,False,True,True,False
4,product_ids,False,True,True,False
5,uuid,False,True,True,False
6,email,False,True,True,False
7,url,False,True,False,False
8,city,False,False,False,False
9,filepath,False,True,True,False


In [63]:
for name in [
    "current",
    "pr903",
    "final",
]:
    y_true = (
        results_df["expected"]
    )

    y_pred = (
        results_df[name]
    )

    print(name)

    print(
        "accuracy:",
        accuracy_score(
            y_true,
            y_pred,
        ),
    )

    print(
        "precision:",
        precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
    )

    print(
        "recall:",
        recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
    )

    print(
        "f1:",
        f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
    )

    print()

current
accuracy: 0.3
precision: 0.2222222222222222
recall: 1.0
f1: 0.36363636363636365

pr903
accuracy: 0.4
precision: 0.25
recall: 1.0
f1: 0.4

final
accuracy: 0.9
precision: 1.0
recall: 0.5
f1: 0.6666666666666666

